In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

In [ ]:
# 3. Create DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [ ]:
# 4. Print shape of one batch
images_batch, ages_batch = next(iter(train_loader))

print("\nOne batch shapes:")
print("images_batch shape:", images_batch.shape)
print("ages_batch shape:", ages_batch.shape)

In [ ]:
# 5. Display sample images
def show_batch_images(images, ages, n=5):
    plt.figure(figsize=(12, 3))
    for i in range(n):
        img = images[i].permute(1, 2, 0).numpy()  # (C, H, W) -> (H, W, C)

        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.title(f"Age: {ages[i].item():.0f}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_batch_images(images_batch, ages_batch, n=5)

In [ ]:
# Task 1: Write your model class here:

class AgeRegressor(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(AgeRegressor, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output linear layer: hidden layer -> 1 number (age)
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines forward pass through network
    def forward(self, x):
        # x shape: (batch_size, input_dim)
        a1 = self.relu(self.layer1(x))
        a2 = self.relu(self.layer2(a1))
        a3 = self.relu(self.layer3(a2))
        output = self.layer4(a3)   # regression output
        return output

In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, optimizer, criterion, train_loader, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move to device
        X_batch = X_batch.to(device)  # (B, 3, 36, 36)
        y_batch = y_batch.to(device)  # (B, 1)

        # Flatten images: (B, 3, 36, 36) -> (B, 3*36*36)
        X_batch = X_batch.view(X_batch.size(0), -1)

        # Forward
        outputs = model(X_batch)      # (B, 1)
        loss = criterion(outputs, y_batch)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Sum loss
        running_loss += loss.item() * X_batch.size(0)

    avg_loss = running_loss / len(train_loader.dataset)
    return avg_loss


In [ ]:
# Task 3: Write your Val loop here:
def validate(model, criterion, test_loader, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Flatten images: (B, 3, 36, 36) -> (B, 3*36*36)
            X_batch = X_batch.view(X_batch.size(0), -1)

            # Forward pass
            outputs = model(X_batch)  # (B, 1)
            loss = criterion(outputs, y_batch)


            running_loss += loss.item() * X_batch.size(0)


    avg_loss = running_loss / len(test_loader.dataset)

    return avg_loss


In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Select device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_dim = 3 * 36 * 36
hidden_dim = 256
output_dim = 1

model = AgeRegressor(input_dim, hidden_dim, output_dim).to(device)

# Regression loss
criterion = nn.MSELoss()

# Optimizer
learning_rate = 0.001
optimizer = AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20

train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')


In [ ]:
# Task 1: Write your code here:

plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss")
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training and Validation Loss over Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:

model.eval()

# Get one batch from test_loader
images_batch, ages_batch = next(iter(test_loader))

# Move to device
images_batch = images_batch.to(device)
ages_batch = ages_batch.to(device)

# Flatten and predict
with torch.no_grad():
    X_flat = images_batch.view(images_batch.size(0), -1)
    preds = model(X_flat)

images_batch_cpu = images_batch.cpu()
ages_cpu = ages_batch.cpu()
preds_cpu = preds.cpu()

# Show first 5 images with true vs predicted age
n = 5
plt.figure(figsize=(12, 3))
for i in range(n):
    img = images_batch_cpu[i].permute(1, 2, 0).numpy()
    true_age = ages_cpu[i].item()
    pred_age = preds_cpu[i].item()

    plt.subplot(1, n, i + 1)
    plt.imshow(img)
    plt.title(f"True: {true_age:.0f}\nPred: {pred_age:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()